# Squaring the rectangle

**CAS BMAI, weekend 1. Friday 10:00, first-order optimization algorithms.**
Exercise `hs26-bmai-w1-opt-heron`. TAs: Jinlei, Daniele, Ivan.

A rectangle has area 2026. You choose its width. The area is fixed, so the
height is whatever makes the area come out to 2026 again.

Somewhere between a tall sliver and a flat slab that rectangle is a square, and
its side is the square root of 2026. You are going to find it without a square
root key, using nothing but plus, minus, times and divide.

You will write four functions:

| | |
|---|---|
| `other_side` | given the width, the height the fixed area forces |
| `verdict` | too small, too large, or exact |
| `heron_step` | one correction |
| `heron` | the loop |

Then a cell you do not have to write prints how many leading digits of your
answer are correct after each correction. It reads **3, 6, 14, 30, 61**. The
count doubles.

Nothing in this notebook needs the network, an API key, or a repository to
clone. It runs offline from the first cell to the last.

In [ ]:
# @title Setup. Run this cell first, then leave it alone.
# There is no pip install, no clone, no API key and no network call anywhere in
# this notebook. Everything below is the Python standard library.
NOTEBOOK_VERSION = "v1.0"

from decimal import Decimal, getcontext

AREA = 2026

print("hs26-bmai-w1-opt-heron, notebook %s" % NOTEBOOK_VERSION)
print("The area is %d." % AREA)
print("45 squared is %d, so a width of 45 misses the area by exactly %d."
      % (45 ** 2, AREA - 45 ** 2))

## Task 1. The other side

The rectangle has area `A` and width `theta`. Return its height.

Whatever you return, multiplying it by `theta` has to give `A` back. That is
the only constraint, and it is the whole of this function.

In [ ]:
# TASK 1 of 4 : other_side
def other_side(theta, A=AREA):
    """The rectangle has area A and width theta. Return its height."""
    # TODO: one line. The area has to come out to A.
    raise NotImplementedError("fill in other_side")


print("other_side(45.0) should be 45.022222222222226")

## Task 2. The verdict

Carlos's third column is `2026 - theta**2`. It is **positive when the guess is
too small**, because then the square you would build is smaller than the area
you were given.

Return the string `'too small'`, `'too large'` or `'exact'`.

The verdict is also readable off the picture, with no arithmetic at all: the
guess is too small exactly when the rectangle is taller than it is wide.

In [ ]:
# TASK 2 of 4 : verdict
def verdict(theta, A=AREA):
    """Return 'too small', 'too large' or 'exact'."""
    # TODO: compare theta squared against A, using the sign convention above.
    raise NotImplementedError("fill in verdict")


print("verdict(45.0) should be 'too small', because 45 squared is 2025")

## Task 3. One correction

You now know two numbers: the width you guessed and the height the area handed
back. The answer is always between them.

With two numbers, no square root key, and the knowledge that the answer sits
somewhere in between, the cheapest thing anybody can do is take the middle.

Return the next width.

In [ ]:
# TASK 3 of 4 : heron_step
def heron_step(theta, A=AREA):
    """One correction: average the two sides of the rectangle."""
    # TODO: average the two sides. Use other_side from task 1.
    raise NotImplementedError("fill in heron_step")


print("heron_step(45.0) should be 45.01111111111111")

## Task 4. The loop

Keep correcting until the square of your width is within `tol` of the area.
Return every width you visited, starting with `theta0`.

Two things to get right, and they are the two places this goes wrong:

1. The stopping test uses the **absolute value** of `A - theta**2`, because the
   residual changes sign after the first correction.
2. `max_steps` is a guard. Without one, a typo in the update turns a ten second
   cell into a hung runtime.

In [ ]:
# TASK 4 of 4 : heron
def heron(A=AREA, theta0=45.0, tol=1e-6, max_steps=50):
    """Correct until |A - theta**2| < tol. Return every width visited."""
    thetas = [theta0]
    # TODO: the loop. Stop on the tolerance or on max_steps.
    raise NotImplementedError("fill in heron")


print("heron(theta0=45.0) should have length 3: the start plus two corrections")

## Check yourself before the interesting part

Run this. It is cheap, it takes a second, and it tells you which of the four
functions is wrong before you go looking in the wrong one.

Every expected value in it is real output of `exercise/reference.py`, which you
can run yourself: it prints each number and then asserts it.

In [ ]:
# Sanity check. Nothing here is yours to edit.
def _sanity():
    assert abs(other_side(45.0) - 45.022222222222226) < 1e-12, "other_side(45.0)"
    assert abs(other_side(10.0) - 202.6) < 1e-12, "other_side(10.0)"
    assert verdict(45.0) == "too small", "verdict(45.0), since 45**2 = 2025"
    assert verdict(46.0) == "too large", "verdict(46.0), since 46**2 = 2116"
    assert abs(heron_step(45.0) - 45.01111111111111) < 1e-12, "heron_step(45.0)"
    assert len(heron(theta0=45.0)) == 3, "heron from 45 takes two corrections"
    assert len(heron(theta0=10.0)) == 7, "heron from 10 takes six corrections"

    # The one that teaches something. After the FIRST correction the rectangle
    # is never tall again: every later width is at or above the square root, so
    # every later square is at or above the area. If this trips, you have most
    # likely written a subtraction somewhere the average belongs.
    xs = heron(theta0=10.0)
    assert all(x * x >= AREA - 1e-9 for x in xs[1:]), \
        "after the first correction the rectangle should never be tall again"


try:
    _sanity()
    print("All four functions pass. Go on to the digit column.")
except NotImplementedError as exc:
    print("Still to do: %s" % exc)
except AssertionError as exc:
    print("AssertionError: %s" % exc)
    print("Fix that one before continuing.")
except Exception as exc:
    print("%s: %s" % (type(exc).__name__, exc))
    print("Fix that one before continuing.")

## The digit column

Now the part worth staying for.

Your `heron` runs in ordinary Python floats, which hold about sixteen digits.
Two corrections from 45 is already past that. So this cell reruns your own
update in exact decimal arithmetic at 220 digits and counts, after each
correction, how many leading digits agree with the true square root.

You do not have to write any of it. Run it and read the last line.

Two warnings live in the code as comments, because anybody who edits this cell
hits both:

- `"%.60f" % some_decimal` converts the Decimal to a float first, so it prints
  sixty digits of a sixteen digit number. Use `Decimal.quantize` and `str()`.
- Comparing `str(45)` against the root undercounts. Pad both to the same number
  of decimal places before comparing.

In [ ]:
# The digit column. Given, and not a task. It reruns YOUR update rule in exact
# decimal arithmetic, so what you are watching is your own algorithm.
getcontext().prec = 220
DA = Decimal(AREA)
ROOT = DA.sqrt()


def _digits_agree(x, ref, places=150):
    # TRAP: quantize BOTH operands to the same number of places first. Without
    # this, str(Decimal(45)) is "45", runs out after two characters, and the
    # honest answer of 3 is reported as 2.
    q = Decimal(1).scaleb(-places)
    a = str(x.quantize(q)).replace("-", "").replace(".", "")
    b = str(ref.quantize(q)).replace("-", "").replace(".", "")
    n = 0
    for ca, cb in zip(a, b):
        if ca != cb:
            break
        n += 1
    return n


# TRAP: never "%.60f" % theta here. That converts to float and prints sixty
# digits of a sixteen digit number, and the column comes out 3, 6, 14, 62, 62.
theta = Decimal(45)
column = []
print("  k   correct digits   theta")
for k in range(5):
    d = _digits_agree(theta, ROOT)
    column.append(d)
    print("  %d   %-14d   %s" % (k, d, str(theta.quantize(Decimal(1).scaleb(-40)))))
    theta = heron_step(theta, DA)      # your function, on Decimals

print("")
print("digit column: %s" % " ".join(str(d) for d in column))
print("sqrt(2026)  = %s" % str(ROOT.quantize(Decimal(1).scaleb(-40))))

## What just happened?

Talk about these. No code.

1. The residual `2026 - theta**2` changes sign once and then never again. Which
   step is the sign change, and what does it say about the shape of every
   rectangle after it?

2. Bisection on the interval from 45 to 46 needs 20 corrections to pin the
   answer to a width of 1e-6. Yours needed 2 from the same start. Both of them
   can only ever evaluate `theta**2`. Where does the extra information come
   from?

3. Your correction was computed exactly. Name the one thing you would have to
   stop knowing for that to become impossible.

Question 3 is the one the next exercise answers. In the genie game the target
can only be sampled through noise, the exact correction stops being available,
and how far to move becomes a number you have to choose. At 11:00 that number
gets its name.

## If you finish early and want to go further with an LLM

Paste this into a new conversation with ChatGPT, Gemini or Claude, followed by
your own code:

> You are a coach for a beginner working on a numerical exercise. Do not write
> the function for me. First diagnose what my code actually does. Then ask me
> one or two targeted questions about the part that is wrong. Then hint at a
> strategy without giving the line. If I am still stuck, implement one small
> piece together with me, that piece only. Never return a complete drop-in
> function unprompted.

That contract is deliberate. A model that hands back the finished function has
answered the question you were supposed to answer, and the exercise takes ten
minutes precisely because the answer is short.

## What is next

You are about to meet the same loop twice more this morning, both times with
something taken away.

| | The target | How far to move |
|---|---|---|
| Heron, just now | a number you can evaluate exactly | the average of the two sides |
| The genie, next | visible only through noise | a step size, and it has to shrink |
| House prices, after that | defined by data rather than by an equation | the same step size, renamed |

At 11:00 the thing being corrected becomes a loss surface, and that step size
gets the name it keeps for the rest of the weekend.